# 05 - chain 2 samples

Makes `c2_train_S.npy` (week 36, seed 0, 10% held out as validation) and the fixed 500k test samples of weeks 41-48 (seed 0), same procedure as chain 1. Paths point to our shared Drive folder.

In [1]:
import os, sys, gc
from pathlib import Path
PROJECT = Path(r"G:\My Drive\CESNET_Project")
os.environ["CESNET_PROJECT"] = str(PROJECT)
sys.path.insert(0, str(PROJECT / "code"))
import numpy as np, pandas as pd
import common as C

need = [35, 36, 40] + list(range(41, 49))
missing = [w for w in need if not C.week_path(w, shared=True).exists()]
print("missing processed weeks:", missing or "none")

# ---- S sample from week 36: same procedure as chain 1 (seed 0, 10% validation held out) ----
S = C.load_json("params.json")["S"]
df = C.load_week(36)
n = len(df)
cnt36 = np.bincount(df["y"].to_numpy(), minlength=10)
names = C.class_names()
print(f"\nweek 36: {n:,} 10-class flows; per class:")
for i in range(10):
    print(f"  {i} {names[i]:17s} {cnt36[i]:>9,}")
perm = np.random.default_rng(0).permutation(n)
n_val = n // 10
val_idx, s_idx = np.sort(perm[:n_val]), np.sort(perm[n_val:n_val + S])
for name, idx in [("c2_w36_val_idx", val_idx), ("c2_train_S", s_idx)]:
    p = C.SAMPLES_DIR / f"{name}.npy"
    if p.exists():
        assert np.array_equal(np.load(p), idx), f"{name} exists with different content!"
    else:
        np.save(p, idx)
print(f"S sample: {len(s_idx):,}, validation: {len(val_idx):,}, overlap: {len(np.intersect1d(s_idx, val_idx))}")
del df; gc.collect()

# ---- fixed test samples, weeks 41-48 (500k, seed 0) ----
rows = []
for w in range(41, 49):
    if w in missing:
        print(f"week {w}: NOT PROCESSED YET, skipped"); continue
    p = C.SAMPLES_DIR / f"c2_test_w{w}.npy"
    df = C.load_week(w); n = len(df)
    if p.exists():
        idx = np.load(p)
    else:
        idx = np.sort(np.random.default_rng(0).choice(n, size=min(500_000, n), replace=False))
        np.save(p, idx)
    c = np.bincount(df["y"].to_numpy()[idx], minlength=10)
    rows.append({"week": w, "flows_10class": n, "test_rows": len(idx),
                 **{f"class_{i}": int(c[i]) for i in range(10)}})
    print(f"week {w}: {n:>9,} flows -> test {len(idx):,}, smallest class {c.min():,} ({names[c.argmin()]})")
    del df; gc.collect()
pd.DataFrame(rows).to_csv(C.RESULTS_DIR / "c2_test_support.csv", index=False)
C.log_event("A6.1: c2_train_S.npy + c2_w36_val_idx.npy (week 36, seed 0, 10% val held out) and "
            f"c2_test_w41..48 (500k, seed 0) created; missing weeks: {missing}", who="Aya")

missing processed weeks: [35]

week 36: 2,525,334 10-class flows; per class:
  0 google-ads          505,832
  1 office365           329,816
  2 google-www          251,587
  3 youtube             242,808
  4 tiktok              262,517
  5 snapchat            216,999
  6 teams               221,820
  7 microsoft-update    195,199
  8 facebook-graph      154,662
  9 spotify             144,094
S sample: 1,000,000, validation: 252,533, overlap: 0
week 41: 2,915,380 flows -> test 500,000, smallest class 24,478 (google-www)
week 42: 2,884,674 flows -> test 500,000, smallest class 25,719 (spotify)
week 43: 2,259,723 flows -> test 500,000, smallest class 25,780 (spotify)
week 44: 2,510,007 flows -> test 500,000, smallest class 19,956 (spotify)
week 45: 2,900,128 flows -> test 500,000, smallest class 25,130 (spotify)
week 46: 2,294,856 flows -> test 500,000, smallest class 25,137 (spotify)
week 47: 2,922,996 flows -> test 500,000, smallest class 25,448 (spotify)
week 48: 2,875,137 flows -> t

In [2]:
print("week 35 on Drive:", C.week_path(35, shared=True).exists())
print("c2_samples_meta.json:", (C.SAMPLES_DIR / "c2_samples_meta.json").exists())
for kind in ("budget", "older", "strat"):
    print(f"c2 {kind} files:", len(list(C.SAMPLES_DIR.glob(f"c2_{kind}_*.npy"))), "(expected 20)")

week 35 on Drive: False
c2_samples_meta.json: True
c2 budget files: 20 (expected 20)
c2 older files: 20 (expected 20)
c2 strat files: 20 (expected 20)
